# ⚖️ Risk Scoring Engine — Multi-Factor Risk Assessment

## Audit Risk Analytics Project

This notebook implements and evaluates a multi-factor risk scoring engine that combines anomaly detection results with transaction characteristics to produce a composite **risk score (0-100)** for each transaction.

### Risk Components
| Weight | Component | Description |
|---|---|---|
| 35% | Anomaly Score | ML model output (IF + LOF ensemble) |
| 25% | Amount Risk | Transaction amount deviation (z-score) |
| 15% | Time Risk | Off-hours activity risk |
| 15% | PCA Risk | Feature-space deviation |
| 10% | Velocity Risk | Transaction frequency patterns |

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded')

## 1. Run Full Pipeline
Execute the complete pipeline: load data, engineer features, detect anomalies, then score risk.

In [ ]:
from src.config import PROCESSED_PARQUET, TARGET_COL
from src.feature_engineering import engineer_all_features
from src.anomaly_model import run_anomaly_detection
from src.risk_scorer import calculate_risk_scores

df = pd.read_parquet(PROCESSED_PARQUET)
df = engineer_all_features(df)
df = run_anomaly_detection(df)
df = calculate_risk_scores(df)

print(f'Final dataset: {len(df):,} rows x {len(df.columns)} columns')

## 2. Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Histogram
axes[0].hist(df['risk_score'], bins=100, color='steelblue',
            edgecolor='black', linewidth=0.3, alpha=0.8)
for threshold, color, label in [(25, '#27ae60', 'Low/Med'),
                                (50, '#f39c12', 'Med/High'),
                                (75, '#e74c3c', 'High/Critical')]:
    axes[0].axvline(threshold, color=color, linestyle='--',
                   linewidth=2, label=label)
axes[0].set_title('Risk Score Distribution', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# By category
cat_counts = df['risk_category'].value_counts()
cat_order = ['low', 'medium', 'high', 'critical']
cat_colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
counts = [cat_counts.get(c, 0) for c in cat_order]
axes[1].bar(cat_order, counts, color=cat_colors,
           edgecolor='black', linewidth=0.5)
axes[1].set_title('Transactions by Risk Category', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Count')
for i, c in enumerate(counts):
    pct = c / len(df) * 100
    axes[1].text(i, c + 500, f'{c:,}' + chr(10) + f'({pct:.1f}%)',
               ha='center', fontsize=10)

# Fraud rate by category
fraud_by_cat = df.groupby('risk_category', observed=True).agg(
    fraud_count=(TARGET_COL, 'sum'),
    total=('Amount', 'count')
).reset_index()
fraud_by_cat['fraud_rate'] = fraud_by_cat['fraud_count'] / fraud_by_cat['total'] * 100
fraud_by_cat['risk_category'] = pd.Categorical(
    fraud_by_cat['risk_category'], categories=cat_order, ordered=True)
fraud_by_cat = fraud_by_cat.sort_values('risk_category')
axes[2].bar(fraud_by_cat['risk_category'].astype(str),
           fraud_by_cat['fraud_rate'], color=cat_colors,
           edgecolor='black', linewidth=0.5)
axes[2].set_title('Fraud Rate by Risk Category', fontweight='bold', fontsize=14)
axes[2].set_ylabel('Fraud Rate (%)')
for i, row in fraud_by_cat.iterrows():
    axes[2].text(list(cat_order).index(row['risk_category']),
               row['fraud_rate'] + 0.1,
               f'{row["fraud_rate"]:.2f}%', ha='center', fontsize=11)

plt.suptitle('Risk Score Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Risk Component Analysis
Visualising how each component contributes to the composite score.

In [ ]:
components = ['anomaly_score', 'amount_risk', 'time_risk', 'pca_risk', 'velocity_risk']
weights = [0.35, 0.25, 0.15, 0.15, 0.10]
labels = ['Anomaly (35%)', 'Amount (25%)', 'Time (15%)', 'PCA (15%)', 'Velocity (10%)']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for idx, (comp, label) in enumerate(zip(components, labels)):
    ax = axes_flat[idx]
    ax.hist(df[comp], bins=80, color='steelblue', alpha=0.7,
           edgecolor='black', linewidth=0.2)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Score (0-100)')
    mean_val = df[comp].mean()
    ax.axvline(mean_val, color='red', linestyle='--',
             label=f'Mean: {mean_val:.1f}')
    ax.legend()

# Component correlation heatmap
corr = df[components + ['risk_score']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
           ax=axes_flat[5], center=0, vmin=-1, vmax=1)
axes_flat[5].set_title('Component Correlations', fontweight='bold')

plt.suptitle('Risk Score Components', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Risk Heatmap — Time vs Amount
Visualising risk concentration by transaction hour and amount bucket — replicating a common audit analytics view.

In [ ]:
heatmap_data = df.groupby(['time_hour_int', 'amount_bucket'], observed=True).agg(
    avg_risk=('risk_score', 'mean'),
    fraud_count=(TARGET_COL, 'sum'),
    total=('Amount', 'count'),
).reset_index()

risk_pivot = heatmap_data.pivot_table(
    index='amount_bucket', columns='time_hour_int',
    values='avg_risk', aggfunc='mean')

bucket_order = ['low', 'medium', 'high', 'very_high']
risk_pivot = risk_pivot.reindex(bucket_order)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(risk_pivot, cmap='YlOrRd', annot=True, fmt='.1f',
           ax=ax, linewidths=0.5, cbar_kws={'label': 'Avg Risk Score'})
ax.set_title('Average Risk Score by Hour x Amount Bucket',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Amount Bucket (Materiality)')

plt.tight_layout()
plt.show()

## 5. Top High-Risk Transactions
Drill-down into the highest-risk flagged transactions — the ones an auditor would review first.

In [ ]:
top_risk = df.nlargest(20, 'risk_score')[[
    'Amount', 'risk_score', 'risk_category',
    'anomaly_score', 'amount_risk', 'time_risk',
    'time_hour_int', 'amount_bucket', 'is_business_hours',
    TARGET_COL,
]].copy()
top_risk.columns = ['Amount', 'Risk Score', 'Risk Category',
                     'Anomaly', 'Amount Risk', 'Time Risk',
                     'Hour', 'Materiality', 'Business Hours', 'Is Fraud']
print('Top 20 Highest-Risk Transactions:')
print(top_risk.to_string())

## 6. Cumulative Fraud Detection Curve
If an auditor reviews transactions in order of risk score (highest first), how quickly do they find fraud? This demonstrates the value of risk-based audit sampling.

In [ ]:
df_sorted = df.sort_values('risk_score', ascending=False).reset_index(drop=True)
df_sorted['cumulative_fraud'] = df_sorted[TARGET_COL].cumsum()
total_fraud = df_sorted[TARGET_COL].sum()
df_sorted['pct_reviewed'] = (df_sorted.index + 1) / len(df_sorted) * 100
df_sorted['pct_fraud_found'] = df_sorted['cumulative_fraud'] / total_fraud * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_sorted['pct_reviewed'], df_sorted['pct_fraud_found'],
        color='#e74c3c', linewidth=2, label='Risk-Based Sampling')
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Random Sampling')

# Key milestones
for pct_review in [1, 5, 10, 20]:
    idx = int(len(df_sorted) * pct_review / 100)
    fraud_found = df_sorted.loc[:idx, TARGET_COL].sum() / total_fraud * 100
    ax.scatter(pct_review, fraud_found, color='#e74c3c', s=80, zorder=5)
    ax.annotate(f'{pct_review}% reviewed' + chr(10) + f'{fraud_found:.1f}% fraud found',
               xy=(pct_review, fraud_found),
               xytext=(pct_review + 5, fraud_found - 5),
               fontsize=10, fontweight='bold',
               arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_title('Cumulative Fraud Detection Curve (Risk-Based vs Random)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('% of Transactions Reviewed')
ax.set_ylabel('% of Fraud Detected')
ax.legend(fontsize=12)
ax.set_xlim(0, 50)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
for pct in [1, 5, 10, 20]:
    idx = int(len(df_sorted) * pct / 100)
    caught = df_sorted.loc[:idx, TARGET_COL].sum()
    print(f'Reviewing top {pct}% ({idx:,} txns) catches '
          f'{int(caught)}/{int(total_fraud)} fraud ({caught/total_fraud*100:.1f}%)')

## 7. Risk SQL Analytics (DuckDB)
Demonstrating audit-style SQL queries against the risk-scored dataset.

In [ ]:
con = duckdb.connect()
con.register('risk_transactions', df)

print('=== Risk Category Summary ===')
result = con.execute("""
    SELECT 
        risk_category,
        COUNT(*) as total,
        SUM(Class) as fraud,
        ROUND(SUM(Class) * 100.0 / COUNT(*), 3) as fraud_rate,
        ROUND(AVG(risk_score), 2) as avg_score,
        ROUND(AVG(Amount), 2) as avg_amount,
        ROUND(SUM(Amount), 2) as total_amount
    FROM risk_transactions
    GROUP BY risk_category
    ORDER BY avg_score DESC
""").fetchdf()
print(result.to_string(index=False))

In [ ]:
print('\n=== Off-Hours High Risk Transactions ===')
result = con.execute("""
    SELECT 
        time_hour_int as hour,
        COUNT(*) as high_risk_txns,
        SUM(Class) as fraud,
        ROUND(AVG(risk_score), 2) as avg_risk,
        ROUND(AVG(Amount), 2) as avg_amount
    FROM risk_transactions
    WHERE risk_category IN ('high', 'critical')
      AND is_business_hours = false
    GROUP BY time_hour_int
    ORDER BY avg_risk DESC
    LIMIT 10
""").fetchdf()
print(result.to_string(index=False))

## 8. Key Findings & Audit Recommendations

### Risk Scoring Performance
- The risk engine correctly concentrates fraud in higher risk categories
- **Risk-based sampling** vastly outperforms random: reviewing the top X% catches much more fraud
- Anomaly score is the strongest individual component

### Audit Recommendations
1. **Prioritise by risk score** — review Critical and High categories first
2. **Off-hours monitoring** — night-time high-risk transactions should trigger alerts
3. **Amount materiality** — very high amount transactions combined with anomaly flags need immediate review
4. **Continuous monitoring** — risk scoring should be applied to new transactions periodically

### Next Steps
- Build interactive Streamlit dashboard for real-time exploration
- Create Power BI report for stakeholder presentation
- Generate automated audit findings report